# Getting Started with ukpyn

Welcome to **ukpyn**, a Python client library for accessing [UK Power Networks Open Data](https://ukpowernetworks.opendatasoft.com/).

This tutorial will walk you through:

1. Installing ukpyn
2. Setting up your API key
3. Creating a client and listing datasets
4. Fetching records from a dataset
5. Exporting data to different formats

Let's get started!

## 1. Installation

You can install ukpyn using pip or uv.

**Using pip:**
```bash
pip install ukpyn
```

**Using uv (recommended for faster installs):**
```bash
uv add ukpyn
```

The package will be installed along with its dependencies (httpx, pydantic).

## 2. Setting Up Your API Key

To access the UK Power Networks Open Data API, you need an API key.

### Getting Your API Key

1. Visit the [UK Power Networks Open Data Portal](https://ukpowernetworks.opendatasoft.com/)
2. Create an account or sign in
3. Go to your account settings
4. Generate a new API key

### Setting the Environment Variable

ukpyn looks for your API key in the `UKPN_API_KEY` environment variable.

**On Linux/macOS:**
```bash
export UKPN_API_KEY="your-api-key-here"
```

**On Windows (Command Prompt):**
```cmd
set UKPN_API_KEY=your-api-key-here
```

**On Windows (PowerShell):**
```powershell
$env:UKPN_API_KEY = "your-api-key-here"
```

**For permanent storage**, add this to your shell profile (`.bashrc`, `.zshrc`) or use a `.env` file with python-dotenv.

**Tip:** You can also pass the API key directly to the client (see below), but using environment variables is more secure and keeps secrets out of your code.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify your API key is set (don't print the actual key!)
if os.getenv("UKPN_API_KEY"):
    print("API key is configured!")
else:
    print("Warning: UKPN_API_KEY environment variable is not set.")
    print("Some API features may be limited without authentication.")

## 3. Creating a Client and Listing Datasets

The `UKPNClient` is an **async** client, which means you need to use `async/await` syntax.

In Jupyter notebooks, you can use `await` directly in cells because Jupyter runs an async event loop.

In [ ]:
from ukpyn import UKPNClient

# Create a client instance
# The API key is automatically loaded from UKPN_API_KEY environment variable
client = UKPNClient()

# Or, pass the API key directly (not recommended for production code)
# client = UKPNClient(api_key="your-api-key-here")

### Listing Available Datasets

Let's see what datasets are available from UK Power Networks.

In [ ]:
# List the first 10 datasets
response = await client.list_datasets(limit=10)

print(f"Total datasets available: {response.total_count}")
print("\nFirst 10 datasets:")
print("-" * 60)

for item in response.datasets:
    dataset = item.dataset
    # Get the title from metadata if available
    title = ""
    if dataset.metas and dataset.metas.default:
        title = dataset.metas.default.get("title", "")
    print(f"ID: {dataset.dataset_id}")
    print(f"Title: {title}")
    print("-" * 60)

# Expected output:
# Total datasets available: 42
#
# First 10 datasets:
# ------------------------------------------------------------
# ID: ukpn-smart-meter-installation-volumes
# Title: Smart Meter Installation Volumes
# ------------------------------------------------------------
# ...

### Searching for Datasets

You can search for datasets by keyword.

In [ ]:
# Search for datasets containing "smart" in their metadata
response = await client.list_datasets(search="smart", limit=5)

print(f"Found {response.total_count} datasets matching 'smart'\n")

for item in response.datasets:
    dataset = item.dataset
    title = ""
    if dataset.metas and dataset.metas.default:
        title = dataset.metas.default.get("title", dataset.dataset_id)
    print(f"- {title} ({dataset.dataset_id})")

### Getting Dataset Details

Get full information about a specific dataset, including its fields.

In [ ]:
# Get details for a specific dataset
# Replace with an actual dataset ID from the list above
dataset_id = "ukpn-smart-meter-installation-volumes"

try:
    dataset = await client.get_dataset(dataset_id)
    
    print(f"Dataset: {dataset.dataset_id}")
    print(f"Has records: {dataset.has_records}")
    
    # Show available fields
    if dataset.fields:
        print(f"\nFields ({len(dataset.fields)}):")
        for field in dataset.fields:
            print(f"  - {field.name} ({field.type}): {field.label or 'No label'}")
            
except Exception as e:
    print(f"Error: {e}")
    print("\nTip: Make sure the dataset_id exists. Check the list_datasets output above.")

## 4. Fetching Records from a Dataset

Once you know which dataset you want, you can fetch its records.

In [ ]:
# Fetch 5 records from the dataset
dataset_id = "ukpn-smart-meter-installation-volumes"  # Change to a valid dataset ID

try:
    records_response = await client.get_records(dataset_id, limit=5)
    
    print(f"Total records in dataset: {records_response.total_count}")
    print(f"Records returned: {len(records_response.records)}")
    print("\n" + "=" * 60)
    
    for i, record in enumerate(records_response.records, 1):
        print(f"\nRecord {i} (ID: {record.id})")
        print("-" * 40)
        if record.fields:
            for key, value in record.fields.items():
                print(f"  {key}: {value}")
                
except Exception as e:
    print(f"Error fetching records: {e}")

### Filtering Records

You can filter records using ODSQL (OpenDataSoft Query Language).

In [ ]:
# Example: Filter records with a WHERE clause
# Note: Adjust the field names and values based on your dataset

try:
    # Filter example - adjust based on actual dataset fields
    filtered_records = await client.get_records(
        dataset_id,
        limit=5,
        # where="field_name = 'value'",  # Uncomment and adjust
        order_by="-record_timestamp",  # Sort by newest first (if available)
    )
    
    print(f"Filtered results: {filtered_records.total_count} total records")
    print(f"Showing first {len(filtered_records.records)} records")
    
except Exception as e:
    print(f"Error: {e}")
    print("\nTip: Check that the field names in your WHERE clause exist in the dataset.")

### Selecting Specific Fields

Use the `select` parameter to return only specific fields.

In [ ]:
# Select only specific fields to reduce response size
try:
    # Adjust field names based on your dataset
    records = await client.get_records(
        dataset_id,
        limit=5,
        # select="field1, field2",  # Uncomment and adjust
    )
    
    print(f"Retrieved {len(records.records)} records")
    
    # Display as a simple table
    for record in records.records:
        if record.fields:
            print(record.fields)
            
except Exception as e:
    print(f"Error: {e}")

### Pagination

For large datasets, use `offset` and `limit` to paginate through results.

In [ ]:
# Paginate through records
page_size = 10
page_number = 0  # 0-indexed

try:
    records = await client.get_records(
        dataset_id,
        limit=page_size,
        offset=page_number * page_size,
    )
    
    total_pages = (records.total_count + page_size - 1) // page_size
    
    print(f"Page {page_number + 1} of {total_pages}")
    print(f"Showing records {page_number * page_size + 1} to {page_number * page_size + len(records.records)}")
    print(f"Total records: {records.total_count}")
    
except Exception as e:
    print(f"Error: {e}")

## 5. Exporting Data to Different Formats

ukpyn supports exporting data in multiple formats:

- `json` - JSON format
- `csv` - Comma-separated values
- `xlsx` - Microsoft Excel format
- `geojson` - GeoJSON (for geographic data)
- `shapefile` - Esri Shapefile (for GIS applications)
- `kml` - Keyhole Markup Language (for Google Earth)

In [ ]:
from ukpyn import EXPORT_FORMATS

print("Available export formats:")
for fmt in EXPORT_FORMATS:
    print(f"  - {fmt}")

### Export to CSV

In [ ]:
# Export dataset to CSV
try:
    from pathlib import Path

    csv_data = await client.export_data(
        dataset_id,
        format="csv",
        limit=100,  # Limit to 100 records for this example
    )

    save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
    if save_dir:
        output_file = Path(save_dir) / "export.csv"
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "wb") as f:
            f.write(csv_data)
        print(f"Exported {len(csv_data)} bytes to {output_file}")
    else:
        print(f"Exported {len(csv_data)} bytes (file save skipped; set save_dir to enable writing).")

    # Preview first few lines
    print("\nPreview (first 500 characters):")
    print(csv_data.decode("utf-8")[:500])

except Exception as e:
    print(f"Export error: {e}")

### Export to JSON

In [ ]:
import json

# Export dataset to JSON
try:
    json_data = await client.export_data(
        dataset_id,
        format="json",
        limit=10,
    )
    
    # Parse and pretty-print
    data = json.loads(json_data)
    print(json.dumps(data[:2], indent=2))  # Show first 2 records
    
except Exception as e:
    print(f"Export error: {e}")

### Export to Excel

In [ ]:
# Export dataset to Excel
try:
    from pathlib import Path

    xlsx_data = await client.export_data(
        dataset_id,
        format="xlsx",
        limit=100,
    )

    save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
    if save_dir:
        output_file = Path(save_dir) / "export.xlsx"
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "wb") as f:
            f.write(xlsx_data)
        print(f"Exported {len(xlsx_data)} bytes to {output_file}")
        print("Open the file in Excel or use pandas to read it.")
    else:
        print(f"Exported {len(xlsx_data)} bytes (file save skipped; set save_dir to enable writing).")

except Exception as e:
    print(f"Export error: {e}")

### Working with pandas (optional)

If you have pandas installed, you can easily convert exports to DataFrames.

In [ ]:
# Optional: Use pandas for data analysis
# Make sure pandas is installed: pip install pandas

try:
    import pandas as pd
    from io import BytesIO
    
    # Export to CSV and load into pandas
    csv_data = await client.export_data(
        dataset_id,
        format="csv",
        limit=100,
    )
    
    # Create DataFrame
    df = pd.read_csv(BytesIO(csv_data), sep=";")
    
    print(f"DataFrame shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    
except ImportError:
    print("pandas is not installed. Install it with: pip install pandas")
except Exception as e:
    print(f"Error: {e}")

## Cleaning Up

When you're done, close the client to release resources.

In [ ]:
# Close the client when done
await client.close()
print("Client closed.")

### Using Context Managers (Recommended)

The preferred way to use UKPNClient is with an async context manager, which automatically handles cleanup.

In [ ]:
# Recommended: Use async context manager for automatic cleanup
async with UKPNClient() as client:
    datasets = await client.list_datasets(limit=3)
    print(f"Found {datasets.total_count} datasets")
    
# Client is automatically closed when exiting the 'async with' block
print("Client automatically closed!")

## Error Handling

ukpyn provides specific exception types for different error conditions.

In [ ]:
from ukpyn import (
    UKPNError,
    AuthenticationError,
    NotFoundError,
    RateLimitError,
    ValidationError,
    ServerError,
)

async with UKPNClient() as client:
    try:
        # Try to access a non-existent dataset
        dataset = await client.get_dataset("this-dataset-does-not-exist")
        
    except NotFoundError as e:
        print(f"Dataset not found: {e}")
        
    except AuthenticationError as e:
        print(f"Authentication failed: {e}")
        print("Tip: Check that your UKPN_API_KEY is correct.")
        
    except RateLimitError as e:
        print(f"Rate limit exceeded: {e}")
        if e.retry_after:
            print(f"Try again in {e.retry_after} seconds.")
            
    except ValidationError as e:
        print(f"Invalid request: {e}")
        
    except ServerError as e:
        print(f"Server error: {e}")
        print("Tip: The UKPN API might be experiencing issues. Try again later.")
        
    except UKPNError as e:
        # Catch-all for other API errors
        print(f"API error: {e}")

## Summary

You've learned how to:

- Install ukpyn using pip or uv
- Configure your API key via environment variable
- Create a UKPNClient and list available datasets
- Search for datasets and view their fields
- Fetch records with filtering, sorting, and pagination
- Export data to CSV, JSON, Excel, and other formats
- Handle errors gracefully

## Next Steps

- Explore the [UK Power Networks Open Data Portal](https://ukpowernetworks.opendatasoft.com/) to discover available datasets
- Learn about [ODSQL](https://help.opendatasoft.com/apis/ods-explore-v2/#section/Opendatasoft-Query-Language-(ODSQL)) for advanced filtering
- Check out the ukpyn documentation for more advanced features

Happy data exploring!